# SignBridge-XMF — Training & studi perbandingan model (Google Colab GPU)

Notebook ini **tidak** menjalankan MediaPipe. Preprocessing sudah dikerjakan di laptop, hasilnya diunggah ke Google Drive dalam bentuk `xmf_processed.zip`.

**Sebelum mulai:**
1. Pilih *Runtime → Change runtime type → T4 GPU*.
2. Pastikan `xmf_processed.zip` sudah ada di Google Drive pada folder `SignBridge/` (isinya `processed/` dan `bank/`).

Semua hasil (checkpoint, tabel, model ONNX) disimpan ke Drive di `SignBridge/outputs`. Kalau Colab terputus, jalankan ulang semua sel dari atas. Run yang sudah selesai akan **dilewati otomatis**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Hubungkan Drive dan ambil kode

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/SignBridge'
BRANCH = 'rombak/xmf-crossmodal-fusion'
!mkdir -p "$DRIVE/outputs"
!rm -rf /content/SignBridge && git clone -q -b $BRANCH https://github.com/KarinaGlo/SignBridge.git /content/SignBridge
%cd /content/SignBridge
!git log --oneline -3

In [ ]:
# torch & torchvision sudah tersedia di Colab; MediaPipe tidak dibutuhkan untuk training
!pip install -q onnx onnxruntime onnxscript pyyaml pandas matplotlib

## 2. Data hasil preprocessing dan folder output di Drive

In [ ]:
import os, glob
!mkdir -p data/xmf && unzip -q -o "$DRIVE/xmf_processed.zip" -d data/xmf
!rm -rf outputs && ln -s "$DRIVE/outputs" outputs
print('gloss npz   :', len(glob.glob('data/xmf/processed/gloss/*.npz')))
print('alfabet npz :', len(glob.glob('data/xmf/processed/alphabet/*.npz')))
print('bank        :', os.listdir('data/xmf/bank'))

In [ ]:
# Percepat: simpan dataset di RAM (gloss ±1 GB) dan pakai 2 worker
import yaml
for t in ('gloss', 'alphabet'):
    p = f'configs/xmf_{t}.yaml'
    c = yaml.safe_load(open(p)); c['cache'] = True; c['num_workers'] = 2
    yaml.safe_dump(c, open(p, 'w'), sort_keys=False, allow_unicode=True)

## 3. Uji cepat (1 epoch) untuk memastikan semuanya jalan

In [ ]:
!python -m signbridge.xmf.train --config configs/xmf_gloss.yaml --protocol sd --epochs 1 --no-eval --out /content/_smoke

## 4. Studi perbandingan — GLOSS (32 kata)

- **Fase A**: 6 fusion (early, weighted, gated, late, refusion, attention) dengan encoder default.
- **Fase B**: fusion terbaik dari fase A, lalu encoder video/citra/audio/teks ditukar satu per satu.

Protokol **SD** (penanda yang sama) dan **SI** (penanda 4 tidak dilihat saat training). Jumlah epoch dibatasi supaya perbandingan muat di sesi Colab. Model final nanti dilatih penuh.

In [ ]:
EPOCHS = 30
!python -m signbridge.xmf.compare --task gloss --protocol sd --phase all --epochs $EPOCHS

In [ ]:
!python -m signbridge.xmf.compare --task gloss --protocol si --phase all --epochs $EPOCHS

In [ ]:
from IPython.display import Markdown, display
for p in ('sd', 'si'):
    f = f'outputs/xmf/compare/gloss_{p}/results.md'
    if os.path.exists(f): display(Markdown(open(f).read()))

## 5. Studi perbandingan — ALFABET (fase A: fusion saja, protokol SD per perangkat)

In [ ]:
!python -m signbridge.xmf.compare --task alphabet --protocol sd --phase A --epochs 25
display(Markdown(open('outputs/xmf/compare/alphabet_sd/results.md').read()))

## 6. Model final untuk deploy

Isi `BEST_SETS` dengan varian terbaik dari tabel **SI** (lihat kolom *variant*). Contoh: `gated-audio_conformer` berarti
`--set model.fusion=gated --set model.encoders.audio=conformer`. Pastikan ukuran ONNX ≲ 25 MB supaya tetap ringan di browser.

In [ ]:
BEST_SETS = '--set model.fusion=attention'          # <- ganti sesuai hasil
!python -m signbridge.xmf.train --config configs/xmf_gloss.yaml --protocol final $BEST_SETS --out outputs/xmf/FINAL_gloss
!python -m signbridge.xmf.export --ckpt outputs/xmf/FINAL_gloss/best.pt --out outputs/web_model

In [ ]:
BEST_SETS_ALPHA = '--set model.fusion=attention'    # <- ganti sesuai hasil alfabet
!python -m signbridge.xmf.train --config configs/xmf_alphabet.yaml --protocol final $BEST_SETS_ALPHA --out outputs/xmf/FINAL_alphabet
!python -m signbridge.xmf.export --ckpt outputs/xmf/FINAL_alphabet/best.pt --out outputs/web_model
!ls -la outputs/web_model

Selesai. Unduh isi `SignBridge/outputs/web_model/` dari Drive (file `xmf_gloss.onnx`, `xmf_gloss.json`, `xmf_alphabet.onnx`, `xmf_alphabet.json`) lalu taruh di folder `web/model/` di laptop. Unduh juga folder `outputs/xmf/compare/` untuk tabel laporan.